# Analytics Layer (dbt) Testing
================================

Test and explore the analytics layer SQL queries used by dbt models.

In [ ]:
# Configuration
import os
import clickhouse_connect

CLICKHOUSE_CONFIG = {
    "host": os.getenv("CLICKHOUSE_HOST", "localhost"),
    "port": os.getenv("CLICKHOUSE_PORT", "8123"),
    "database": os.getenv("CLICKHOUSE_DB", "insurance_db"),
    "username": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", "clickhouse_pass"),
}

ch_client = clickhouse_connect.get_client(
    host=CLICKHOUSE_CONFIG["host"],
    port=CLICKHOUSE_CONFIG["port"],
    database=CLICKHOUSE_CONFIG["database"],
    username=CLICKHOUSE_CONFIG["username"],
    password=CLICKHOUSE_CONFIG["password"],
)

print("Connected to ClickHouse")

## Current State

In [ ]:
# List all tables and views
result = ch_client.query("SHOW TABLES")
print("=== Tables & Views in ClickHouse ===\n")
for row in sorted(result.result_rows):
    print(f"  {row[0]}")

## View: analytics_customers

In [ ]:
# Test analytics_customers view
print("=== analytics_customers ===\n")

result = ch_client.query("SELECT * FROM analytics_customers LIMIT 5")
print("Sample rows:")
for row in result.result_rows:
    print(row)

result = ch_client.query("SELECT count() FROM analytics_customers")
print(f"\nTotal rows: {result.result_rows[0][0]}")

In [ ]:
# Risk bucket distribution
result = ch_client.query("""
    SELECT 
        risk_bucket, 
        count() as customers,
        round(100.0 * count() / sum(count()) OVER (), 1) as pct
    FROM analytics_customers
    GROUP BY risk_bucket
    ORDER BY customers DESC
""")
print("Risk bucket distribution:")
for row in result.result_rows:
    print(f"  {row[0]}: {row[1]} ({row[2]}%)")

## View: analytics_claims

In [ ]:
# Test analytics_claims view
print("=== analytics_claims ===\n")

result = ch_client.query("SELECT * FROM analytics_claims LIMIT 5")
print("Sample rows:")
for row in result.result_rows:
    print(row)

result = ch_client.query("SELECT count() FROM analytics_claims")
print(f"\nTotal rows: {result.result_rows[0][0]}")

In [ ]:
# Total claim amounts
result = ch_client.query("""
    SELECT 
        sum(claim_amount) as total_amount,
        sum(claim_paid_amount) as total_paid,
        sum(claim_amount) - sum(claim_paid_amount) as outstanding
    FROM analytics_claims
""")
print("Claim amounts:")
row = result.result_rows[0]
print(f"  Total amount: ${row[0]:,.0f}")
print(f"  Total paid: ${row[1]:,.0f}")
print(f"  Outstanding: ${row[2]:,.0f}")

## View: analytics_claims_by_status

In [ ]:
# Test analytics_claims_by_status view
print("=== analytics_claims_by_status ===\n")

result = ch_client.query("SELECT * FROM analytics_claims_by_status ORDER BY total_amount DESC")
print("Claims by status:")
for row in result.result_rows:
    print(f"  {row[0]}: {row[1]} claims, ${row[2]:,.0f}")

## View: analytics_claims_by_business_line

In [ ]:
# Test analytics_claims_by_business_line view
print("=== analytics_claims_by_business_line ===\n")

result = ch_client.query("SELECT * FROM analytics_claims_by_business_line ORDER BY total_amount DESC")
print("Claims by business line (vehicle category):")
for row in result.result_rows:
    print(f"  {row[0]}: {row[1]} claims, ${row[2]:,.0f}")

## View: analytics_claims_by_agent

In [ ]:
# Test analytics_claims_by_agent view
print("=== analytics_claims_by_agent ===\n")

result = ch_client.query("SELECT * FROM analytics_claims_by_agent ORDER BY total_amount DESC LIMIT 10")
print("Top 10 agents by claim amount:")
for row in result.result_rows:
    print(f"  {row[0]}: {row[1]} claims, ${row[2]:,.0f}")

## Run dbt Models

In [ ]:
# Run dbt to rebuild analytics
import subprocess

result = subprocess.run(
    ["dbt", "run", "--target", "clickhouse"],
    cwd="../dbt",
    capture_output=True,
    text=True,
)
print("=== dbt run output ===")
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("stderr:", result.stderr[-500:])

## Verify After dbt Run

In [ ]:
# Verify all analytics views after rebuild
print("=== Analytics Views After dbt ===\n")

views = [
    "analytics_customers",
    "analytics_claims",
    "analytics_claims_by_status",
    "analytics_claims_by_agent",
    "analytics_claims_by_business_line",
]

for view in views:
    try:
        result = ch_client.query(f"SELECT count() FROM {view}")
        count = result.result_rows[0][0]
        print(f"  {view}: {count:,} rows")
    except Exception as e:
        print(f"  {view}: ERROR - {e}")

## SQL: analytics_customers

In [ ]:
# Print the raw SQL
print("=== analytics_customers SQL ===\n")
with open("../dbt/models/analytics/analytics_customers.sql", "r") as f:
    print(f.read())

## SQL: analytics_claims

In [ ]:
print("=== analytics_claims SQL ===\n")
with open("../dbt/models/analytics/analytics_claims.sql", "r") as f:
    print(f.read())

## SQL: analytics_claims_by_status

In [ ]:
print("=== analytics_claims_by_status SQL ===\n")
with open("../dbt/models/analytics/analytics_claims_by_status.sql", "r") as f:
    print(f.read())

## SQL: analytics_claims_by_business_line

In [ ]:
print("=== analytics_claims_by_business_line SQL ===\n")
with open("../dbt/models/analytics/analytics_claims_by_business_line.sql", "r") as f:
    print(f.read())

## SQL: analytics_claims_by_agent

In [ ]:
print("=== analytics_claims_by_agent SQL ===\n")
with open("../dbt/models/analytics/analytics_claims_by_agent.sql", "r") as f:
    print(f.read())